## LSTM for text generation based on Pushkin's texts

In [5]:
with open('pushkin.txt') as f:
    text_database = f.read()

CYRILLIC_SYMBOLS = set('абвгдеёжзийклмнопрстуфхцчшщъыьэюяєіїґ')
PUNCTUATION_SYMBOLS = set('!"#$%&\'()*+,-./:;<=>?@[\\]_`{|}~«»… \t\n')

def normalize_text(text):
    """Replace various dash characters with regular hyphen"""
    # Replace all types of dashes with regular hyphen
    dashes = ['–', '—', '−', '‒', '―', '‐', '‑']
    for dash in dashes:
        text = text.replace(dash, '-')
    return text.replace('\xa0', ' ').strip()

all_lines = text_database.split('\n')
verse_lines = [
    normalize_text(line)
    for line in all_lines 
    # All verses are starting with two tabs in this text corpus
    if line.startswith('\t\t')
]

training_lines = [
    line.lower()
    for line in verse_lines 
    # Remove verses that contain non-cyrillic symbols and non-punctuation symbols
    if all(char.lower() in CYRILLIC_SYMBOLS or char in PUNCTUATION_SYMBOLS for char in line)
        # Some verses are empty or really small, let's get rid of them
        and len(line) >= 5
]
training_data = "\n".join(training_lines)

character_set = sorted(list(set(training_data)))
vocabulary_size = len(character_set)
print("Character set:", f"{vocabulary_size} chars", "".join(character_set))

verse_lengths = [len(verse) for verse in training_lines]
print(f"Average length: {sum(verse_lengths) / len(verse_lengths):.1f} chars")
print(f"Min length: {min(verse_lengths)}")
print(f"Max length: {max(verse_lengths)}")
print(f"Median length: {sorted(verse_lengths)[len(verse_lengths)//2]}")
print("Total lines:", len(training_lines))

print("\nSample text:\n" + training_data[:100])

Character set: 54 chars 
 !"'()*,-.:;<>?[]«»абвгдежзийклмнопрстуфхцчшщъыьэюяё…
Average length: 27.4 chars
Min length: 5
Max length: 58
Median length: 26
Total lines: 20712

Sample text:
скажите мне, почему «похититель»
освистан партером?
увы, потому что бедный автор
похитил его у молье


In [6]:
char_to_index = {u:i for i, u in enumerate(character_set)}
index_to_char = {i:u for i, u in enumerate(character_set)}

print(char_to_index)
print(index_to_char)

{'\n': 0, ' ': 1, '!': 2, '"': 3, "'": 4, '(': 5, ')': 6, '*': 7, ',': 8, '-': 9, '.': 10, ':': 11, ';': 12, '<': 13, '>': 14, '?': 15, '[': 16, ']': 17, '«': 18, '»': 19, 'а': 20, 'б': 21, 'в': 22, 'г': 23, 'д': 24, 'е': 25, 'ж': 26, 'з': 27, 'и': 28, 'й': 29, 'к': 30, 'л': 31, 'м': 32, 'н': 33, 'о': 34, 'п': 35, 'р': 36, 'с': 37, 'т': 38, 'у': 39, 'ф': 40, 'х': 41, 'ц': 42, 'ч': 43, 'ш': 44, 'щ': 45, 'ъ': 46, 'ы': 47, 'ь': 48, 'э': 49, 'ю': 50, 'я': 51, 'ё': 52, '…': 53}
{0: '\n', 1: ' ', 2: '!', 3: '"', 4: "'", 5: '(', 6: ')', 7: '*', 8: ',', 9: '-', 10: '.', 11: ':', 12: ';', 13: '<', 14: '>', 15: '?', 16: '[', 17: ']', 18: '«', 19: '»', 20: 'а', 21: 'б', 22: 'в', 23: 'г', 24: 'д', 25: 'е', 26: 'ж', 27: 'з', 28: 'и', 29: 'й', 30: 'к', 31: 'л', 32: 'м', 33: 'н', 34: 'о', 35: 'п', 36: 'р', 37: 'с', 38: 'т', 39: 'у', 40: 'ф', 41: 'х', 42: 'ц', 43: 'ч', 44: 'ш', 45: 'щ', 46: 'ъ', 47: 'ы', 48: 'ь', 49: 'э', 50: 'ю', 51: 'я', 52: 'ё', 53: '…'}


In [7]:
import torch
import torch.nn as nn

class LstmLayer(nn.Module):
    def __init__(self, input_size, output_size, num_classes):
        super().__init__()
        self.input_size = input_size
        self.output_size = output_size
        self.num_classes = num_classes
        self.stacked_size = input_size + output_size
        
        # Smaller initial gradients for better stability
        self.Wc = nn.Parameter(torch.randn(self.stacked_size, self.output_size) * 0.01)
        self.bc = nn.Parameter(torch.zeros(self.output_size))

        self.Wu = nn.Parameter(torch.randn(self.stacked_size, self.output_size) * 0.01)
        self.bu = nn.Parameter(torch.zeros(self.output_size))

        self.Wf = nn.Parameter(torch.randn(self.stacked_size, self.output_size) * 0.01)
        self.bf = nn.Parameter(torch.zeros(self.output_size))

        self.Wo = nn.Parameter(torch.randn(self.stacked_size, self.output_size) * 0.01)
        self.bo = nn.Parameter(torch.zeros(self.output_size))

        # Fully connected layer for softmax output
        self.Wy = nn.Parameter(torch.randn(self.output_size, self.num_classes) * 0.01)
        self.by = nn.Parameter(torch.zeros(self.num_classes))
        
    def forward(self, x, a_prev, c_prev):
        stacked = torch.cat([a_prev, x], dim=1)
        c_hat = torch.tanh(stacked @ self.Wc + self.bc)
        Gu = torch.sigmoid(stacked @ self.Wu + self.bu)
        Gf = torch.sigmoid(stacked @ self.Wf + self.bf)
        Go = torch.sigmoid(stacked @ self.Wo + self.bo)
        c = Gu * c_hat + Gf * c_prev
        a = Go * torch.tanh(c)
        y_logits = a @ self.Wy + self.by
        return a, c, y_logits

In [ ]:
num_epochs = 100
batch_size = 64

temporal_depth = 3
# Each training example is a sequence of length temporal_depth
num_training_exampes = len(training_data) - temporal_depth
X = [training_data[m:m+temporal_depth] for m in range(num_training_exampes)]

# True value for the next predicted characters is Y, same as X shifted by 1 to the left
shifted_training_data = training_data[1:] + "\n"
Y = [shifted_training_data[m:m+temporal_depth] for m in range(num_training_exampes)]

# Input size is vocabulary size because each character is one-hot encoded
# output_size is the hidden layer activation/memory size, we can experiment with it
layers = [
    LstmLayer(input_size=vocabulary_size, output_size=128, num_classes=vocabulary_size) 
    for i in range(temporal_depth)
]

losses = []
for epoch in range(num_epochs):
    for batch_start in range(0, num_training_exampes, batch_size):
        current_batch_size = min(batch_size, num_training_exampes - batch_start)
        X_batch = X[batch_start:batch_start+current_batch_size]
        Y_batch = Y[batch_start:batch_start+current_batch_size]

        X_batch_onehot = torch.zeros(current_batch_size, vocabulary_size)

        a_prev = torch.zeros(num_training_exampes, current_batch_size)
        for layer in layers:

            # 1. FORWARD PASS
            predictions = layers(X)  # Calls forward() automatically
            predictions = predictions.squeeze()  # (100, 1) → (100,)
            
            # 2. COMPUTE LOSS
            loss = loss_fn(predictions, y)
            
            # 3. BACKWARD PASS (compute gradients)
            optimizer.zero_grad()  # Clear old gradients
            loss.backward()        # Compute new gradients (autograd magic!)
            
            # 4. UPDATE WEIGHTS
            optimizer.step()       # Update W and b using gradients
            
            # Track loss
            losses.append(loss.item())
            
            # Print every 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

# Plot loss curve
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()